In [1]:
import pandas as pd
import numpy as np
import os
from scipy.optimize import minimize

scores_df = pd.read_csv("df_preparados/scores_df.csv", index_col=0)
returns = pd.read_csv("df_preparados/returns.csv", index_col=0, parse_dates=True)

scores_df = scores_df.reset_index()

if "ticker" not in scores_df.columns:
    scores_df = scores_df.rename(columns={"index": "ticker"})


def optimize_portfolio(returns, min_weight=0.03, max_weight=0.60):

    mean_returns = returns.mean() * 252
    cov_matrix = returns.cov() * 252

    n_assets = len(mean_returns)

    def portfolio_volatility(weights):
        return np.sqrt(weights.T @ cov_matrix @ weights)

    def negative_sharpe(weights):
        portfolio_return = np.dot(weights, mean_returns)
        portfolio_risk = portfolio_volatility(weights)

        if portfolio_risk == 0:
            return 999

        return -(portfolio_return / portfolio_risk)

    constraints = (
        {"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    bounds = tuple((min_weight, max_weight) for _ in range(n_assets))

    initial_weights = np.array([1 / n_assets] * n_assets)

    result = minimize(
        negative_sharpe,
        initial_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        print("Optimization warning:", result.message)

    return pd.Series(
        result.x,
        index=returns.columns
    )


strategies = {
    "dividend": "dividend_score",
    "growth": "growth_score",
    "value": "value_score",
    "momentum": "momentum_score",
    "low_volatility": "low_vol_score"
}

portfolio_dfs = {}

for strategy_name, score_column in strategies.items():

    print(f"\nCreating {strategy_name} portfolio")

    top_assets = (
        scores_df
        .dropna(subset=[score_column])
        .sort_values(score_column, ascending=False)
        .head(10)
    )

    tickers = top_assets["ticker"].tolist()

    tickers = [ticker for ticker in tickers if ticker in returns.columns]

    selected_returns = returns[tickers].dropna()

    weights = optimize_portfolio(
        selected_returns,
        min_weight=0.03,
        max_weight=0.60
    )

    portfolio_df = pd.DataFrame({
        "ticker": weights.index,
        "weight": weights.values
    })

    portfolio_df = portfolio_df.sort_values(
        "weight",
        ascending=False
    )

    portfolio_dfs[strategy_name] = portfolio_df

    print(portfolio_df)


os.makedirs("carteras", exist_ok=True)

for strategy_name, df in portfolio_dfs.items():

    file_path = f"carteras/{strategy_name}_portfolio.csv"

    df.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")


Creating dividend portfolio
        ticker    weight
5  RELIANCE.NS  0.447286
3      ASML.AS  0.240086
4         AAPL  0.096460
1          JNJ  0.080432
0        MC.PA  0.075737
6         BABA  0.030000
2      NESN.SW  0.030000

Creating growth portfolio
        ticker    weight
3  RELIANCE.NS  0.425382
1      ASML.AS  0.226558
6        BRK-B  0.085615
0         AAPL  0.058741
2          JPM  0.057356
8        MC.PA  0.056348
4          JNJ  0.030000
7      NESN.SW  0.030000
5         BABA  0.030000

Creating value portfolio
        ticker    weight
3  RELIANCE.NS  0.447286
5      ASML.AS  0.240086
4         AAPL  0.096460
2          JNJ  0.080432
1        MC.PA  0.075737
0      NESN.SW  0.030000
6         BABA  0.030000

Creating momentum portfolio
        ticker    weight
6          TIP  0.563429
5  RELIANCE.NS  0.164111
0      ASML.AS  0.062460
9      ETH-USD  0.030000
8        BRK-B  0.030000
1          JNJ  0.030000
4          JPM  0.030000
3      NESN.SW  0.030000
2         AAPL